In [ ]:
import vrep 
import sys
import time 
import numpy as np
from tank import *
import skfuzzy
from skfuzzy import control as ctrl

In [ ]:
def find_place(dist_NW, dist_WN, dist_NE):
    wn = ctrl.Antecedent(np.arange(0, 5.01, 0.01), 'WN')
    nw = ctrl.Antecedent(np.arange(0, 6.01, 0.01), 'NW')
    ne = ctrl.Antecedent(np.arange(0, 6.01, 0.01), 'NE')
    speed = ctrl.Consequent(np.arange(-1, 7.01, 0.01), 'speed')

    # Funkcje przynależności
    wn['close'] = fuzz.trapmf(wn.universe, [0, 0, 1.5, 3])
    wn['far']   = fuzz.trapmf(wn.universe, [1.5, 2.5, 5, 5])

    nw['close'] = fuzz.trapmf(nw.universe, [0, 0, 1.5, 2])
    nw['far']   = fuzz.trapmf(nw.universe, [1, 2, 4, 5])

    ne['close'] = fuzz.trapmf(ne.universe, [0, 0, 1.5, 2])
    ne['far']   = fuzz.trapmf(ne.universe, [1, 2, 4, 5])

    speed['stop']  = fuzz.trimf(speed.universe, [-1, 0, 1])
    speed['break'] = fuzz.trimf(speed.universe, [0, 2, 4])
    speed['go']    = fuzz.trimf(speed.universe, [3, 5, 7])

    rules = [
        ctrl.Rule(wn['close'] & nw['close'] & ne['close'], speed['go']),
        ctrl.Rule(wn['far'] & nw['close'] & ne['far'], speed['stop']),
        ctrl.Rule(wn['close'] & nw['far'], speed['go']),
        ctrl.Rule(wn['far'] & nw['far'], speed['go'])
    ]

    sim_ctrl = ctrl.ControlSystem(rules)
    sim = ctrl.ControlSystemSimulation(sim_ctrl)

    sim.input['WN'] = dist_WN
    sim.input['NW'] = dist_NW
    sim.input['NE'] = dist_NE

    sim.compute()
    return sim.output['speed']


def park(dist_WN, dist_EN, dist_WS, dist_ES):
    wn = ctrl.Antecedent(np.arange(0, 5.01, 0.01), 'WN')
    en = ctrl.Antecedent(np.arange(0, 6.01, 0.01), 'EN')
    ws = ctrl.Antecedent(np.arange(0, 6.01, 0.01), 'WS')
    es = ctrl.Antecedent(np.arange(0, 6.01, 0.01), 'ES')
    speed = ctrl.Consequent(np.arange(-1, 5.01, 0.01), 'speed')

    # Funkcje przynależności
    wn['close'] = fuzz.trapmf(wn.universe, [0, 0, 1, 2])
    wn['far']   = fuzz.trapmf(wn.universe, [1.5, 2.5, 5, 5])

    en['close'] = fuzz.trapmf(en.universe, [0, 0, 1.5, 2])
    en['far']   = fuzz.trapmf(en.universe, [1, 2, 3, 6])

    ws['close'] = fuzz.trapmf(ws.universe, [0, 0, 1.5, 2])
    ws['far']   = fuzz.trapmf(ws.universe, [1, 2, 3, 6])

    es['close'] = fuzz.trapmf(es.universe, [0, 0, 1.5, 2])
    es['far']   = fuzz.trapmf(es.universe, [1, 2, 3, 6])

    speed['stop']  = fuzz.trimf(speed.universe, [-1, 0, 1])
    speed['break'] = fuzz.trimf(speed.universe, [0, 1, 2])
    speed['go']    = fuzz.trimf(speed.universe, [1, 2, 5])

    rules = [
        ctrl.Rule(wn['far'] & en['far'], speed['go']),
        ctrl.Rule(wn['far'] & en['far'], speed['break']),
        ctrl.Rule(wn['close'] & en['close'], speed['break']),
        ctrl.Rule(wn['close'] & en['far'], speed['break']),

        ctrl.Rule(ws['far'] & es['close'], speed['go']),
        ctrl.Rule(ws['far'] & es['far'], speed['break']),
        ctrl.Rule(ws['close'] & es['close'], speed['stop']),
        ctrl.Rule(ws['close'] & es['far'], speed['break']),
    ]

    sim_ctrl = ctrl.ControlSystem(rules)
    sim = ctrl.ControlSystemSimulation(sim_ctrl)

    sim.input['WN'] = dist_WN
    sim.input['EN'] = dist_EN
    sim.input['WS'] = dist_WS
    sim.input['ES'] = dist_ES

    sim.compute()
    return sim.output['speed']


In [118]:
vrep.simxFinish(-1) # closes all opened connections, in case any prevoius wasnt finished
clientID=vrep.simxStart('127.0.0.1',19999,True,True,5000,5) # start a connection

if clientID!=-1:
    print ("Connected to remote API server")
else:
    print("Not connected to remote API server")
    sys.exit("Could not connect")

#create instance of Tank
tank=Tank(clientID)

Connected to remote API server


In [119]:
proximity_sensors=["EN","ES","NE","NW","SE","SW","WN","WS"]
proximity_sensors_handles=[0]*8

# get handle to proximity sensors
for i in range(len(proximity_sensors)):
    err_code,proximity_sensors_handles[i] = vrep.simxGetObjectHandle(clientID,"Proximity_sensor_"+proximity_sensors[i], vrep.simx_opmode_blocking)
    
#read and print values from proximity sensors
#first reading should be done with simx_opmode_streaming, further with simx_opmode_buffer parameter
for sensor_name, sensor_handle in zip(proximity_sensors,proximity_sensors_handles):
        err_code,detectionState,detectedPoint,detectedObjectHandle,detectedSurfaceNormalVector=vrep.simxReadProximitySensor(clientID,sensor_handle,vrep.simx_opmode_streaming)

In [ ]:
tank.forward(5)

distances = dict()
detection_states = dict()

for sensor_name, sensor_handle in zip(proximity_sensors,proximity_sensors_handles):
    err_code,detectionState,detectedPoint,detectedObjectHandle,detectedSurfaceNormalVector=vrep.simxReadProximitySensor(clientID,sensor_handle,vrep.simx_opmode_buffer)
    distances[sensor_name] = np.linalg.norm(detectedPoint)
    detection_states[sensor_name] = detectionState

stages = [
     'find_place',
     'park',
     'go_forward'
]
counter = 0
stage = 'find_place'
#continue reading and printing values from proximity sensors
t = time.time()
while (time.time()-t)<100: # read values for 5 seconds
    counter += 1
    for sensor_name, sensor_handle in zip(proximity_sensors,proximity_sensors_handles):
        err_code,detectionState,detectedPoint,detectedObjectHandle,detectedSurfaceNormalVector=vrep.simxReadProximitySensor(clientID,sensor_handle,vrep.simx_opmode_buffer )
        if(err_code == 0):
            # print("Proximity_sensor_"+sensor_name, np.linalg.norm(detectedPoint))
            distances[sensor_name] = np.linalg.norm(detectedPoint)
            detection_states[sensor_name] = detectionState
    
    if stage == 'find_place':
        tank_speed = find_place(distances['NW'], distances['WN'], distances['NE'])
        if tank_speed < 3.45:
            tank.stop()
            stage = 'park'
            print("Zmiana na fazę parkowania")
        else: 
            tank.forward(tank_speed)
    if stage == 'park':
        tank_speed = park(distances['WN'], distances['EN'], distances['WS'], distances['ES'])
        if counter % 2000:
            # print(distances['WN'], distances['EN'], distances['WS'], distances['ES'])
            print(tank_speed)
        if tank_speed < 0.51:
            tank.forward(1)
            stage = 'go_forward'
            print("Faza wyrównania")
        else:
            tank.leftvelocity = tank_speed
            tank.rightvelocity = tank_speed * 2
            tank.setVelocity()
            # print(tank.leftvelocity, tank.rightvelocity)
            tank.go()
    if stage == 'go_forward':
        tank.forward(1)
        if not detection_states['NW'] and not detection_states['NE']:
            tank.stop()
            break


    # print()

Zmiana na fazę parkowania
2.591290983900869
2.591290983900869
2.591290983900869
2.591290983900869
2.594451245439165
2.594451245439165
2.594451245439165
2.594451245439165
2.594451245439165
2.597471437000643
2.597471437000643
2.597471437000643
2.597471437000643
2.597471437000643
2.597492393166078
2.597492393166078
2.597492393166078
2.597492393166078
2.597492393166078
2.6463172082485453
2.6463172082485453
2.6463172082485453
2.6463172082485453
2.6463172082485453
2.6081967388945078
2.6081967388945078
2.6081967388945078
2.6081967388945078
2.60597653395969
2.60597653395969
2.60597653395969
2.60597653395969
2.60597653395969
2.298727409357552
2.298727409357552
2.298727409357552
2.298727409357552
2.298727409357552
2.298694226636461
2.298694226636461
2.298694226636461
2.298694226636461
2.298694226636461
2.2979926934664707
2.2979926934664707
2.2979926934664707
2.2979926934664707
2.2979926934664707
2.292805416208577
2.292805416208577
2.292805416208577
2.292805416208577
2.2861466922349236
2.28614669